# Data Preparation: 46-Class Fruit Classification (fru92 subset)

Select the first 50% of classes (alphabetically) from the fru92 dataset and split into Training/Validation/Test sets.

In [ ]:
import os
import shutil
import random
import zipfile
from pathlib import Path

# Configuration
ZIP_PATH = Path("/home/azureuser/cloudfiles/code/Users/p62984/AIS.ba-VZ/GAI4/GAI4UE/fru92_images.zip")
DATA_DIR = Path("data")
SOURCE_BASE = DATA_DIR / "fru92_images"
TARGET_BASE = Path("PrepData")
SPLITS = ["Training", "Validation", "Test"]
SPLIT_RATIOS = (0.7, 0.15, 0.15)  # train, val, test
SEED = 42

# Extract zip if not already extracted
if not SOURCE_BASE.exists():
    print(f"Extracting {ZIP_PATH.name} to {DATA_DIR}/...")
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall(DATA_DIR)
    print("Extraction complete.")
else:
    print(f"Dataset already extracted at {SOURCE_BASE}")

# Get all class folders sorted alphabetically, take first 50%
all_classes = sorted([d.name for d in SOURCE_BASE.iterdir() if d.is_dir()])
n_select = len(all_classes) // 2
selected_classes = all_classes[:n_select]
print(f"Total classes: {len(all_classes)}, selected (first 50%): {len(selected_classes)}")
print(f"Classes: {selected_classes}")

def prepare_dataset(classes, source_base, target_base):
    if target_base.exists():
        print(f"Removing existing directory: {target_base}")
        shutil.rmtree(target_base)

    rng = random.Random(SEED)

    for cls_name in classes:
        src_dir = source_base / cls_name
        images = sorted([p for p in src_dir.glob("*") if p.suffix.lower() in [".jpg", ".jpeg", ".png"]])
        rng.shuffle(images)

        n = len(images)
        n_train = int(n * SPLIT_RATIOS[0])
        n_val = int(n * SPLIT_RATIOS[1])
        split_images = {
            "Training": images[:n_train],
            "Validation": images[n_train:n_train + n_val],
            "Test": images[n_train + n_val:],
        }

        for split in SPLITS:
            dest_dir = target_base / split / cls_name
            dest_dir.mkdir(parents=True, exist_ok=True)
            for img_path in split_images[split]:
                os.symlink(img_path.absolute(), dest_dir / img_path.name)
            print(f"  {split}/{cls_name}: {len(split_images[split])} images")

prepare_dataset(selected_classes, SOURCE_BASE, TARGET_BASE)